# Face/place stimulus selection - per subject + shared

NSD has 8 subjects. Each saw 1,000 images shared with every other subject,
plus ~9,000 images unique to them alone. This notebook builds, for each
subject, a 400-image set:

- **200 shared** - 100 faces + 100 places, the same for every subject
  (since these are the images all 8 subjects saw)
- **200 unique** - 100 faces + 100 places seen only by that one subject

900 distinct faces and 900 distinct places in total (100 shared + 800
unique across the 8 subjects, in each category). Images are saved into
separate folders by source and category, e.g.
`stimulus_subset/faces/subject3/` or `stimulus_subset/places/shared/`.

Fully automated: a face is anything the YuNet detector finds a face in; a
place is anything with no detected face and a Places365 (ResNet-18)
scene-confidence above 0.5. No manual review step.

In [1]:
import ast
import io
import os
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
import requests
import torch
import torchvision
from PIL import Image
from torchvision import transforms

BUCKET = "https://natural-scenes-dataset.s3.amazonaws.com"
PLACE_CONFIDENCE_THRESHOLD = 0.5
N_SHARED_PER_CATEGORY = 100
N_UNIQUE_PER_SUBJECT_PER_CATEGORY = 100

os.makedirs("models", exist_ok=True)
SOURCES = ["shared"] + [f"subject{i}" for i in range(1, 9)]
for category in ["faces", "places"]:
    for source in SOURCES:
        os.makedirs(f"stimulus_subset/{category}/{source}", exist_ok=True)

## 1. YuNet & Places365 (ResNet-18) setup

In [2]:
yunet_path = "models/face_detection_yunet_2023mar.onnx"
if not os.path.exists(yunet_path):
    r = requests.get(
        "https://github.com/opencv/opencv_zoo/raw/main/models/"
        "face_detection_yunet/face_detection_yunet_2023mar.onnx"
    )
    r.raise_for_status()
    with open(yunet_path, "wb") as f:
        f.write(r.content)
face_detector = cv2.FaceDetectorYN_create(yunet_path, "", (425, 425))

ckpt_path = "models/resnet18_places365.pth.tar"
if not os.path.exists(ckpt_path):
    r = requests.get("http://places2.csail.mit.edu/models_places365/resnet18_places365.pth.tar")
    r.raise_for_status()
    with open(ckpt_path, "wb") as f:
        f.write(r.content)

cats_path = "models/categories_places365.txt"
if not os.path.exists(cats_path):
    r = requests.get("https://raw.githubusercontent.com/CSAILVision/places365/master/categories_places365.txt")
    r.raise_for_status()
    with open(cats_path, "w") as f:
        f.write(r.text)
place_categories = [line.split(" ")[0][3:] for line in open(cats_path)]

device = "cuda" if torch.cuda.is_available() else "cpu"
places_model = torchvision.models.resnet18(num_classes=365)
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
state_dict = {k.replace("module.", ""): v for k, v in checkpoint["state_dict"].items()}
places_model.load_state_dict(state_dict)
places_model.eval().to(device)

places_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def detect_face(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    _, faces = face_detector.detect(bgr)
    return 0 if faces is None else len(faces)

def classify_scene(img):
    x = places_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(places_model(x), dim=1)[0].cpu().numpy()
    top1_idx = int(probs.argmax())
    return place_categories[top1_idx], float(probs[top1_idx])

print(f"both detectors ready, Places365 on {device}")


both detectors ready, Places365 on cuda


## 2. NSD's per-image metadata

One row per image: which of the 8 subjects saw it (`subject1`-`subject8`),
whether it's part of shared1000, its COCO id/crop info. Everything below
is built from this one table.

In [3]:
stim_info = pd.read_csv(f"{BUCKET}/nsddata/experiments/nsd/nsd_stim_info_merged.csv", index_col=0)
print(f"images: {len(stim_info)}")

images: 73000


## 3. Fetching images

Shared1000 images are already NSD-cropped
PNGs on S3, ready to use. Every other image comes straight from COCO as
a raw rectangular photo, and needs NSD's own crop reconstructed
(`cropBox`: the top/bottom/left/right fractions NSD cropped away) so what
gets checked matches what a subject actually saw.

In [4]:
resp = requests.get(
    BUCKET, params={"list-type": "2", "prefix": "nsddata/stimuli/nsd/shared1000/", "max-keys": "2000"}
)
resp.raise_for_status()
root = ET.fromstring(resp.content)
ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
key_by_nsd_id = {}
for obj in root.findall("s3:Contents", ns):
    key = obj.find("s3:Key", ns).text
    nid = int(key.rsplit("/", 1)[-1].split("_nsd")[1].split(".")[0]) - 1
    key_by_nsd_id[nid] = key

def fetch_shared_image(nsd_id):
    img_bytes = requests.get(f"{BUCKET}/{key_by_nsd_id[nsd_id]}").content
    return Image.open(io.BytesIO(img_bytes)).convert("RGB")

def crop_nsd_style(img, crop_box_str, out_size=425):
    top, bottom, left, right = ast.literal_eval(crop_box_str)
    w, h = img.size
    if w >= h:
        box = (round(left * w), 0, w - round(right * w), h)
    else:
        box = (0, round(top * h), w, h - round(bottom * h))
    return img.crop(box).resize((out_size, out_size), Image.LANCZOS)

def fetch_coco_image(row):
    url = f"http://images.cocodataset.org/{row['cocoSplit']}/{row['cocoId']:012d}.jpg"
    img_bytes = requests.get(url).content
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    return crop_nsd_style(img, row["cropBox"])


## 4. The shared 100 faces + 100 places


In [5]:
shared_rows = stim_info[stim_info["shared1000"]].copy()
shared_images = {row["nsdId"]: fetch_shared_image(row["nsdId"]) for _, row in shared_rows.iterrows()}
print(f"downloaded {len(shared_images)} shared1000 images")

shared_labels = []
for nsd_id, img in shared_images.items():
    n_faces = detect_face(img)
    top1_scene, confidence = classify_scene(img)
    shared_labels.append({"nsdId": nsd_id, "n_faces": n_faces, "has_face": n_faces > 0,
                           "top1_scene": top1_scene, "confidence": confidence})
shared_labels = pd.DataFrame(shared_labels)
print(f"real faces in shared1000: {shared_labels['has_face'].sum()}")


downloaded 1000 shared1000 images


real faces in shared1000: 114


In [6]:
shared_face_ids = (
    shared_labels[shared_labels["has_face"]]
    .sample(frac=1, random_state=0)["nsdId"]
    .head(N_SHARED_PER_CATEGORY).tolist()
)
shared_place_ids = (
    shared_labels[~shared_labels["has_face"]]
    .sort_values("confidence", ascending=False)
    .head(N_SHARED_PER_CATEGORY)["nsdId"].tolist()
)

shared_faces = [{"nsdId": nid, "image": shared_images[nid], "source": "shared"} for nid in shared_face_ids]
shared_places = [{"nsdId": nid, "image": shared_images[nid], "source": "shared"} for nid in shared_place_ids]
print(f"shared faces: {len(shared_faces)}, shared places: {len(shared_places)}")


shared faces: 100, shared places: 100


## 5. Per-subject: 100 faces + 100 places

For each subject's own ~9,000-image pool (seen only by them)

In [7]:
from concurrent.futures import ThreadPoolExecutor

subject_faces = {}
subject_places = {}
BATCH_SIZE = 32

for subj in range(1, 9):
    unique_pool = stim_info[(stim_info[f"subject{subj}"] == 1) & (~stim_info["shared1000"])]

    # --- faces: keep checking until 100 found ---
    face_pool = unique_pool.sample(frac=1, random_state=subj).reset_index(drop=True)
    found_faces = []
    i = 0
    while len(found_faces) < N_UNIQUE_PER_SUBJECT_PER_CATEGORY and i < len(face_pool):
        batch = face_pool.iloc[i:i + BATCH_SIZE]
        i += BATCH_SIZE
        with ThreadPoolExecutor(max_workers=16) as pool:
            imgs = list(pool.map(fetch_coco_image, [row for _, row in batch.iterrows()]))
        for (_, row), img in zip(batch.iterrows(), imgs):
            if detect_face(img) > 0:
                found_faces.append({"nsdId": int(row["nsdId"]), "image": img, "source": f"subject{subj}"})
                if len(found_faces) >= N_UNIQUE_PER_SUBJECT_PER_CATEGORY:
                    break
    subject_faces[subj] = found_faces

    # --- places: any remaining unique image, no face + confident scene label ---
    used_ids = {item["nsdId"] for item in found_faces}
    place_pool = unique_pool[~unique_pool["nsdId"].isin(used_ids)]
    place_pool = place_pool.sample(frac=1, random_state=subj + 100).reset_index(drop=True)
    found_places = []
    i = 0
    while len(found_places) < N_UNIQUE_PER_SUBJECT_PER_CATEGORY and i < len(place_pool):
        batch = place_pool.iloc[i:i + BATCH_SIZE]
        i += BATCH_SIZE
        with ThreadPoolExecutor(max_workers=16) as pool:
            imgs = list(pool.map(fetch_coco_image, [row for _, row in batch.iterrows()]))
        for (_, row), img in zip(batch.iterrows(), imgs):
            if detect_face(img) > 0:
                continue
            top1_scene, confidence = classify_scene(img)
            if confidence > PLACE_CONFIDENCE_THRESHOLD:
                found_places.append({"nsdId": int(row["nsdId"]), "image": img, "source": f"subject{subj}"})
                if len(found_places) >= N_UNIQUE_PER_SUBJECT_PER_CATEGORY:
                    break
    subject_places[subj] = found_places

    print(f"subject {subj}: {len(found_faces)} faces, {len(found_places)} places")

subject 1: 100 faces, 100 places


subject 2: 100 faces, 100 places


subject 3: 100 faces, 100 places


subject 4: 100 faces, 100 places


subject 5: 100 faces, 100 places


subject 6: 100 faces, 100 places


subject 7: 100 faces, 100 places


subject 8: 100 faces, 100 places


## 6. Saving the datasets

In [8]:
all_faces = shared_faces + [item for items in subject_faces.values() for item in items]
all_places = shared_places + [item for items in subject_places.values() for item in items]

print(f"total faces: {len(all_faces)}, total places: {len(all_places)}")
assert len(all_faces) == len(all_places) == N_SHARED_PER_CATEGORY + 8 * N_UNIQUE_PER_SUBJECT_PER_CATEGORY
assert len(set(item["nsdId"] for item in all_faces)) == len(all_faces)
assert len(set(item["nsdId"] for item in all_places)) == len(all_places)
assert not set(item["nsdId"] for item in all_faces) & set(item["nsdId"] for item in all_places)

for item in all_faces:
    item["image"].save(f"stimulus_subset/faces/{item['source']}/face_nsd{item['nsdId']:05d}.png")
for item in all_places:
    item["image"].save(f"stimulus_subset/places/{item['source']}/place_nsd{item['nsdId']:05d}.png")
print(f"saved {len(all_faces)} images under stimulus_subset/faces/<source>/")
print(f"saved {len(all_places)} images under stimulus_subset/places/<source>/")


total faces: 900, total places: 900


saved 900 images under stimulus_subset/faces/<source>/
saved 900 images under stimulus_subset/places/<source>/


## 7. Saving the image list

In [9]:
image_list = pd.DataFrame(
    [{"nsdId": item["nsdId"], "category": "face", "source": item["source"]} for item in all_faces]
    + [{"nsdId": item["nsdId"], "category": "place", "source": item["source"]} for item in all_places]
).sort_values(["category", "source", "nsdId"]).reset_index(drop=True)

os.makedirs("data", exist_ok=True)
image_list.to_csv("data/nsd_face_place_stimulus_subset.csv", index=False)
print(image_list.groupby(["category", "source"]).size())
print("saved: data/nsd_face_place_stimulus_subset.csv")


category  source  
face      shared      100
          subject1    100
          subject2    100
          subject3    100
          subject4    100
          subject5    100
          subject6    100
          subject7    100
          subject8    100
place     shared      100
          subject1    100
          subject2    100
          subject3    100
          subject4    100
          subject5    100
          subject6    100
          subject7    100
          subject8    100
dtype: int64
saved: data/nsd_face_place_stimulus_subset.csv


## Summary

- Selected 900 face images and 900 place images: 100 of each shared by
  all 8 subjects, plus 100 of each unique to every individual subject
  (8 x 100 = 800 per category).
- Faces come from NSD's full 73k pool, confirmed by the YuNet face
  detector. Places come from the same pools, accepted when the face
  detector finds nothing and a Places365-trained ResNet18 scores its top
  scene category above 0.5 confidence.
- Saved under `stimulus_subset/faces/<source>/` and
  `stimulus_subset/places/<source>/`, one subfolder per source (`shared`,
  `subject1`-`subject8`), 100 images each - 900 total per category.
  `data/nsd_face_place_stimulus_subset.csv` records each image's category
  and source.